# 🎤 STT Comparison: Whisper vs Gemini API

**Metrics**: Latency | BLEU Score | Token Usage (from API)

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================

AUDIO_FILE = "f3f2abf1-266f-4bf2-ad77-4a19fdf8cb57.webm"

# Whisper (self-hosted)
WHISPER_MODEL_SIZE = "base"
WHISPER_DEVICE = "cpu"
WHISPER_COMPUTE_TYPE = "int8"

# Gemini API pricing (USD per 1M tokens)
GEMINI_2_FLASH_INPUT_PRICE = 0.70
GEMINI_2_FLASH_OUTPUT_PRICE = 0.4

GEMINI_2_LITE_INPUT_PRICE = 0.075
GEMINI_2_LITE_OUTPUT_PRICE = 0.3


In [ ]:
import sys, os, time, math
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ai_service_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.extend([project_root, ai_service_root])

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

from faster_whisper import WhisperModel
from google import genai
from google.genai import types
from src.core.config import settings

AUDIO_PATH = os.path.join(project_root, "server", "static", "recordings", AUDIO_FILE)
REFERENCE_PATH = os.path.join(ai_service_root, "src", "agents", "meeting_to_task", "meeting_audio", "transcript.txt")

with open(REFERENCE_PATH, 'r', encoding='utf-8') as f:
    REFERENCE_TRANSCRIPT = f.read().strip()

client = genai.Client(api_key=settings.google_key)

print(f"Audio: {AUDIO_FILE}")
print(f"Reference: {len(REFERENCE_TRANSCRIPT)} chars")

In [ ]:
def calculate_bleu(reference: str, candidate: str, max_n: int = 4) -> float:
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()
    if len(cand_tokens) == 0: return 0.0
    
    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter([tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1)])
        cand_ngrams = Counter([tuple(cand_tokens[i:i+n]) for i in range(len(cand_tokens) - n + 1)])
        matches = sum(min(count, ref_ngrams.get(ngram, 0)) for ngram, count in cand_ngrams.items())
        total = sum(cand_ngrams.values())
        precisions.append(matches / total if total > 0 else 0)
    
    bp = 1.0 if len(cand_tokens) >= len(ref_tokens) else math.exp(1 - len(ref_tokens) / len(cand_tokens))
    return (bp * math.exp(sum(math.log(p) for p in precisions) / len(precisions)) if min(precisions) > 0 else 0.0) * 100

def transcribe_gemini(audio_path: str, model: str) -> dict:
    """Transcribe with Gemini and return transcript + actual token usage"""
    start = time.time()
    
    # Upload file
    upload = client.files.upload(file=audio_path)
    while client.files.get(name=upload.name).state.name != "ACTIVE":
        time.sleep(1)
    
    file_info = client.files.get(name=upload.name)
    
    # Transcribe - use text= keyword argument
    response = client.models.generate_content(
        model=model,
        contents=[types.Content(parts=[
            types.Part.from_text(text="Transcribe audio chính xác. Chỉ output transcript, không thêm gì khác."),
            types.Part.from_uri(file_uri=file_info.uri, mime_type=file_info.mime_type)
        ])]
    )
    
    latency = time.time() - start
    
    # Get actual token usage from response
    usage = response.usage_metadata
    input_tokens = usage.prompt_token_count if usage else 0
    output_tokens = usage.candidates_token_count if usage else 0
    
    # Cleanup
    try: client.files.delete(name=upload.name)
    except: pass
    
    return {
        "transcript": response.text,
        "latency": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens
    }

## Run Experiments

In [ ]:
# Whisper
print("Running Whisper...")
whisper_model = WhisperModel(WHISPER_MODEL_SIZE, device=WHISPER_DEVICE, compute_type=WHISPER_COMPUTE_TYPE)
whisper_start = time.time()
segments, info = whisper_model.transcribe(AUDIO_PATH, language="vi", beam_size=3)
whisper_transcript = " ".join([seg.text for seg in segments])
whisper_latency = time.time() - whisper_start
whisper_duration = info.duration
print(f"Done: {whisper_latency:.2f}s | Audio: {whisper_duration:.1f}s")

Running Whisper...
Done: 20.33s | Audio: 210.3s


In [ ]:
# Gemini 2.0 Flash
print("Running Gemini 2.0 Flash...")
gemini_flash = transcribe_gemini(AUDIO_PATH, "gemini-2.0-flash")
print(f"Done: {gemini_flash['latency']:.2f}s | Tokens: {gemini_flash['input_tokens']} in, {gemini_flash['output_tokens']} out")

Running Gemini 2.0 Flash...
Done: 16.35s | Tokens: 66694 in, 728 out


In [ ]:
# Gemini 2.0 Flash-Lite
print("Running Gemini 2.0 Flash-Lite...")
gemini_lite = transcribe_gemini(AUDIO_PATH, "gemini-2.0-flash-lite")
print(f"Done: {gemini_lite['latency']:.2f}s | Tokens: {gemini_lite['input_tokens']} in, {gemini_lite['output_tokens']} out")

Running Gemini 2.0 Flash-Lite...
Done: 14.46s | Tokens: 66694 in, 731 out


## Experimental Results

In [ ]:
# Calculate BLEU scores
whisper_bleu = calculate_bleu(REFERENCE_TRANSCRIPT, whisper_transcript)
flash_bleu = calculate_bleu(REFERENCE_TRANSCRIPT, gemini_flash['transcript'])
lite_bleu = calculate_bleu(REFERENCE_TRANSCRIPT, gemini_lite['transcript'])

# Calculate costs from actual token usage
flash_cost = (gemini_flash['input_tokens'] * GEMINI_2_FLASH_INPUT_PRICE + gemini_flash['output_tokens'] * GEMINI_2_FLASH_OUTPUT_PRICE) / 1_000_000

lite_cost = (gemini_lite['input_tokens'] * GEMINI_2_LITE_INPUT_PRICE + gemini_lite['output_tokens'] * GEMINI_2_LITE_OUTPUT_PRICE) / 1_000_000


In [ ]:
from IPython.display import display, HTML

html = f"""
<style>
    .t {{ font-family: Arial; border-collapse: collapse; width: 100%; margin: 10px 0; }}
    .t th {{ background: #374151; color: white; padding: 8px; text-align: center; }}
    .t td {{ padding: 8px; text-align: center; border-bottom: 1px solid #e5e7eb; }}
    .m {{ font-weight: 600; text-align: left !important; }}
</style>

<h3>Results</h3>
<table class="t">
<tr><th>Metric</th><th>Whisper (base)</th><th>Gemini 2.0 Flash</th><th>Gemini 2.0 Flash-Lite</th></tr>
<tr><td class="m">Latency</td><td>{whisper_latency:.2f}s</td><td>{gemini_flash['latency']:.2f}s</td><td>{gemini_lite['latency']:.2f}s</td></tr>
<tr><td class="m">BLEU Score</td><td>{whisper_bleu:.2f}%</td><td>{flash_bleu:.2f}%</td><td>{lite_bleu:.2f}%</td></tr>
<tr><td class="m">Transcript Length</td><td>{len(whisper_transcript):,} chars</td><td>{len(gemini_flash['transcript']):,} chars</td><td>{len(gemini_lite['transcript']):,} chars</td></tr>
</table>

<h3>Token Usage & Cost (Gemini only)</h3>
<table class="t">
<tr><th>Metric</th><th>Gemini 2.0 Flash</th><th>Gemini 2.0 Flash-Lite</th></tr>
<tr><td class="m">Input Tokens</td><td>{gemini_flash['input_tokens']:,}</td><td>{gemini_lite['input_tokens']:,}</td></tr>
<tr><td class="m">Output Tokens</td><td>{gemini_flash['output_tokens']:,}</td><td>{gemini_lite['output_tokens']:,}</td></tr>
<tr><td class="m">Total Tokens</td><td>{gemini_flash['total_tokens']:,}</td><td>{gemini_lite['total_tokens']:,}</td></tr>
<tr><td class="m">Cost (USD)</td><td>${flash_cost:.6f}</td><td>${lite_cost:.6f}</td></tr>
</table>

<h3>Configuration</h3>
<table class="t">
<tr><th></th><th>Whisper</th><th>Gemini Flash</th><th>Gemini Flash-Lite</th></tr>
<tr><td class="m">Model</td><td>{WHISPER_MODEL_SIZE}</td><td>gemini-2.0-flash</td><td>gemini-2.0-flash-lite</td></tr>
<tr><td class="m">Device</td><td>{WHISPER_DEVICE} ({WHISPER_COMPUTE_TYPE})</td><td>Cloud</td><td>Cloud</td></tr>
<tr><td class="m">Cost Model</td><td>Self-hosted</td><td>${GEMINI_2_FLASH_INPUT_PRICE}/1M tokens</td><td>${GEMINI_2_LITE_INPUT_PRICE}/1M tokens</td></tr>
</table>

<p><b>Audio:</b> {AUDIO_FILE} ({whisper_duration:.1f}s)</p>
"""

display(HTML(html))